# 04 — Regularisation: L1, L2, and Dropout

## What this notebook covers

Regularisation techniques prevent **overfitting** — when the network memorises training data
instead of learning to generalise. We cover three approaches:

| Technique | What it does | Typical use |
|-----------|-------------|-------------|
| **L2 (weight decay)** | Penalises large weights by adding $\lambda \|W\|^2$ to the loss | Default first choice |
| **L1** | Penalises with $\lambda \|W\|_1$, encourages exact zeros | When you want sparse weights |
| **Dropout** | Randomly zeros neurons during training | Large networks, overfitting |

---

## The math

**L2 loss penalty:** $L_{total} = L_{data} + \lambda \sum_{i,j} w_{ij}^2$

Gradient addition: $\dfrac{\partial L_{total}}{\partial w} = \dfrac{\partial L_{data}}{\partial w} + 2\lambda w$

**L1 loss penalty:** $L_{total} = L_{data} + \lambda \sum_{i,j} |w_{ij}|$

Gradient addition: $\dfrac{\partial L_{total}}{\partial w} = \dfrac{\partial L_{data}}{\partial w} + \lambda \cdot \text{sign}(w)$

**Dropout:** During forward pass, multiply activations by a random binary mask $m \sim \text{Bernoulli}(p)$, scaled by $\frac{1}{p}$ (inverted dropout).


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import nnfs
from nnfs.datasets import spiral_data
nnfs.init()

from core import DenseLayer, DropoutLayer, ReLU, SoftmaxWithCrossEntropy, Adam

X, y           = spiral_data(samples=100, classes=3)
X_test, y_test = spiral_data(samples=100, classes=3)


## Helper: generic train + eval loop

In [ ]:
def run_experiment(name, dense1, dense2, dropout=None, epochs=10001):
    activation1 = ReLU()
    loss_fn     = SoftmaxWithCrossEntropy()
    optimizer   = Adam(learning_rate=0.02, decay=5e-5)

    train_loss, train_acc = [], []

    for epoch in range(epochs):
        # Forward
        dense1.forward(X)
        activation1.forward(dense1.output)
        inp = activation1.output
        if dropout:
            dropout.forward(inp, training=True)
            inp = dropout.output
        dense2.forward(inp)

        data_loss = loss_fn.forward(dense2.output, y)
        reg_loss  = loss_fn.regularization_loss(dense1) + loss_fn.regularization_loss(dense2)
        loss      = data_loss + reg_loss

        preds = np.argmax(loss_fn.output, axis=1)
        acc   = np.mean(preds == y)
        train_loss.append(loss); train_acc.append(acc)

        # Backward
        loss_fn.backward(loss_fn.output, y)
        dense2.backward(loss_fn.dinputs)
        if dropout:
            dropout.backward(dense2.dinputs)
            activation1.backward(dropout.dinputs)
        else:
            activation1.backward(dense2.dinputs)
        dense1.backward(activation1.dinputs)

        optimizer.pre_update_params()
        optimizer.update_params(dense1)
        optimizer.update_params(dense2)
        optimizer.post_update_params()

    # Evaluate on test set (no dropout)
    dense1.forward(X_test)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    test_loss = loss_fn.forward(dense2.output, y_test)
    test_preds = np.argmax(loss_fn.output, axis=1)
    test_acc   = np.mean(test_preds == y_test)

    print(f'{name:<20} | train acc {train_acc[-1]:.3f} | test acc {test_acc:.3f} | test loss {test_loss:.4f}')
    return train_loss, train_acc


## Experiment 1 — No regularisation (baseline)

In [ ]:
np.random.seed(0)
d1 = DenseLayer(2, 64)
d2 = DenseLayer(64, 3)
h_none = run_experiment('No regularisation', d1, d2)


## Experiment 2 — L2 regularisation

In [ ]:
np.random.seed(0)
d1 = DenseLayer(2, 64, l2_reg=5e-4)
d2 = DenseLayer(64, 3, l2_reg=5e-4)
h_l2 = run_experiment('L2 (λ=5e-4)', d1, d2)


## Experiment 3 — L1 regularisation

In [ ]:
np.random.seed(0)
d1 = DenseLayer(2, 64, l1_reg=5e-4)
d2 = DenseLayer(64, 3, l1_reg=5e-4)
h_l1 = run_experiment('L1 (λ=5e-4)', d1, d2)


## Experiment 4 — Dropout (5 %)

In [ ]:
np.random.seed(0)
d1  = DenseLayer(2, 64)
d2  = DenseLayer(64, 3)
drop = DropoutLayer(drop_rate=0.05)
h_drop = run_experiment('Dropout (5%)', d1, d2, dropout=drop)


## Plot: training loss for each experiment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels  = ['No reg', 'L2', 'L1', 'Dropout']
colors  = ['#95a5a6', '#e74c3c', '#3498db', '#2ecc71']
hists   = [h_none, h_l2, h_l1, h_drop]

for (loss, acc), label, color in zip(hists, labels, colors):
    axes[0].plot(loss, label=label, color=color)
    axes[1].plot(acc,  label=label, color=color)

axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Training Accuracy'); axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Regularisation Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


## Key takeaways

- **No regularisation** — highest training accuracy, but potentially worse on unseen data
- **L2** — smooth weight penalty; keeps all weights small, best general choice
- **L1** — drives many weights to exactly zero; useful for feature selection
- **Dropout** — forces the network to learn redundant representations; acts as an ensemble of many networks

> In practice, **L2 + Dropout** together is a very strong combination for most tasks.
